# Quantum Reservoir Computing for Realized Volatility Forecasting — walkthrough

Reproduction of [arXiv:2505.13933](https://arxiv.org/abs/2505.13933).

This notebook is a short executable tour of the reproduction. It

1. rebuilds the authors' dataset from the published normalised panel,
2. runs the 10-qubit quantum reservoir (QR1) and checks it against the authors'
   own saved forecasts,
3. shows the two baseline defects that reverse the paper's conclusion, and
4. runs the MerLin photonic counterpart.

Full results, tables and caveats are in `README.md`; the authoritative workflow
record is `LOG.md`. Run it from the paper folder; total runtime is about a minute.


In [1]:
import logging
import sys
from pathlib import Path

import numpy as np

PAPER_DIR = Path.cwd() if (Path.cwd() / "lib").is_dir() else Path.cwd().parent
REPO_ROOT = PAPER_DIR.parents[1]
for path in (str(REPO_ROOT), str(PAPER_DIR)):
    if path not in sys.path:
        sys.path.insert(0, path)
logging.disable(logging.INFO)  # the library logs run evidence; keep the notebook quiet

DATA_DIR = REPO_ROOT / "data" / "qrc_volatility"
assert (DATA_DIR / "Data.CSV").exists(), f"place the authors' data in {DATA_DIR} (see README)"
print("data:", DATA_DIR)


data: /reproduced_papers/data/qrc_volatility


## 1. Dataset

The authors publish only the min-max normalised panel `Data.CSV`; their raw
`Data_raw.csv` is missing upstream. Raw `log RV` is recovered exactly by inverting
the `Min_RV`/`Max_RV` constants hard-coded in their `Time_series.jl`.


In [2]:
from lib.data import (
    build_lagged_inputs, build_regressor_frame, denormalise_log_rv,
    load_coupling_instances, load_normalised_table, rolling_windows,
)

normalised = load_normalised_table(DATA_DIR)
frame = build_regressor_frame(normalised)          # raw log RV + ADF-differenced exogenous
couplings = load_coupling_instances(DATA_DIR)      # the authors' 100 saved J matrices

N_OUT_OF_SAMPLE, N_LAGS = 245, 3
train_slices, origins = rolling_windows(len(normalised), N_OUT_OF_SAMPLE)
target = normalised["RV"].to_numpy()               # normalised to [-1, 0]
actual = frame["RV"].to_numpy()[np.asarray(origins)]

print(f"{len(normalised)} monthly rows, {normalised.index[0].date()} .. {normalised.index[-1].date()}")
print(f"{N_OUT_OF_SAMPLE} forecasts, {normalised.index[origins[0]].date()} .. {normalised.index[origins[-1]].date()}")
print(f"couplings {couplings.shape}, spectral radius of instance 0 = {np.linalg.eigvalsh(couplings[0]).max():.3f}")
print(f"log RV range: {actual.min():.3f} .. {actual.max():.3f}")


816 monthly rows, 1950-01-31 .. 2017-12-31
245 forecasts, 1997-08-31 .. 2017-12-31
couplings (100, 10, 10), spectral radius of instance 0 = 1.000
log RV range: -4.403 .. -1.430


## 2. The quantum reservoir (QR1)

10 qubits under a fixed transverse-field Ising Hamiltonian: 7 input qubits carry
`RY(pi * x)`-encoded features, 3 hidden qubits carry memory. Three lags are
encoded in turn; between lags the input qubits are traced out. Only the ridge
readout on the ten `<Z_j>` expectations is trained.


In [3]:
from lib.metrics import mse, qlike
from lib.qrc import reservoir_readout, rolling_ridge_forecast

QR1_FEATURES = ["RV", "MKT", "DP", "IP", "RV_q", "STR", "DEF"]   # paper Sec. IV.D
windows = build_lagged_inputs(normalised, QR1_FEATURES, N_LAGS)

readout = reservoir_readout(windows, couplings[0], n_qubits=10, tau=1.0, virtual_nodes=1)
qr1 = denormalise_log_rv(
    rolling_ridge_forecast(readout, target, train_slices, origins, delta=1e-8)
)

print(f"readout matrix {readout.shape}  (one <Z_j> per qubit)")
print(f"QR1  MSE   = {mse(qr1, actual):.4f}   (paper 0.105)")
print(f"QR1  QLIKE = {qlike(qr1, actual):.4f}   (paper 1.4427)")


readout matrix (816, 10)  (one <Z_j> per qubit)
QR1  MSE   = 0.1051   (paper 0.105)
QR1  QLIKE = 1.4427   (paper 1.4427)


### Cross-check against the authors' own output

The upstream repository ships `predict_result.csv`, the 245 forecasts that
produced paper Table II. The reference simulation is `ComplexF32`, so agreement
at `1e-3` means the two implementations are the same computation.


In [4]:
import pandas as pd

reference = pd.read_csv(DATA_DIR / "authors_qr_predictions.csv")["QR1"].to_numpy()
print(f"max |ours - authors| = {np.abs(qr1 - reference).max():.2e}")
print(f"correlation          = {np.corrcoef(qr1, reference)[0, 1]:.12f}")


max |ours - authors| = 6.97e-05
correlation          = 0.999999997389


## 3. Defect 1 — the published HAR/HARX losses are misaligned by one month

The authors' notebook predicts from `dff.iloc[end:end+1]` but scores against
`dff.iloc[-245:]`, so every HAR-family forecast is compared with the wrong month.
The AR, ARMAX, LSTM, classical-reservoir and quantum paths are all correctly
aligned, so only the HAR family is affected. Correcting it changes nothing about
the window and introduces no look-ahead.


In [5]:
from lib.baselines import HARX_EXOGENOUS, har_forecasts

rows = []
for label, kwargs in [
    ("HAR  (as published)", dict(reference_misalignment=True)),
    ("HAR  (corrected)",    dict()),
    ("HARX (as published)", dict(exogenous=HARX_EXOGENOUS, window_start=277,
                                 reference_misalignment=True)),
    ("HARX (corrected)",    dict(exogenous=HARX_EXOGENOUS, window_start=277)),
]:
    forecast = har_forecasts(frame, train_slices, origins, (1,), **kwargs)[1]
    rows.append({"model": label, "MSE": mse(forecast, actual),
                 "QLIKE": qlike(forecast, actual)})
rows.append({"model": "QR1 (quantum)", "MSE": mse(qr1, actual),
             "QLIKE": qlike(qr1, actual)})
pd.DataFrame(rows).round(4)


,model,MSE,QLIKE
0,HAR (as published),0.1477,2.0432
1,HAR (corrected),0.1157,1.5688
2,HARX (as published),0.1448,2.0245
3,HARX (corrected),0.1016,1.3709
4,QR1 (quantum),0.1051,1.4427


A correctly indexed HARX — plain OLS on three HAR terms and four lagged
macro-financial regressors — beats the quantum reservoir on both metrics.

## 4. Defect 2 — the reported quantum numbers are ~5th-percentile draws

The paper reports the best of 100 reservoir instances. Below we score the first
25 of the authors' own saved coupling matrices; the full 100-instance sweep is
`configs/instance_sweep.json` and its result is in `README.md`.


In [6]:
scores = []
for instance in range(25):
    matrix = reservoir_readout(windows, couplings[instance], n_qubits=10, tau=1.0,
                               virtual_nodes=1)
    forecast = denormalise_log_rv(
        rolling_ridge_forecast(matrix, target, train_slices, origins, delta=1e-8)
    )
    scores.append(mse(forecast, actual))
scores = np.array(scores)

print(f"25 instances: mean {scores.mean():.4f} +/- {scores.std(ddof=1):.4f}, "
      f"median {np.median(scores):.4f}, min {scores.min():.4f}")
print(f"the paper's reported QR1 value 0.1050 sits at percentile "
      f"{100 * (scores < 0.1050).mean():.0f} of this sample")
print("full 100-instance sweep (README): mean 0.1095 +/- 0.0026, 5th percentile 0.1051")


25 instances: mean 0.1094 +/- 0.0030, median 0.1094, min 0.1038
the paper's reported QR1 value 0.1050 sits at percentile 12 of this sample
full 100-instance sweep (README): mean 0.1095 +/- 0.0026, 5th percentile 0.1051


Giving a classical control the *same* best-of-100 budget reverses the ranking: an
echo state network with exactly 10 or 20 readout units, the same seven features,
the same three-step window and the same rolling ridge scores 0.1009 and 0.0974
(README, Table II). A plain ridge on the 21 raw lagged features scores 0.1031.

## 5. The MerLin photonic counterpart

10 modes, 3 photons, a frozen Haar mesh, three sequential angle-encoding blocks,
and a per-mode photon-number readout — the direct photonic analogue of the
per-qubit `<Z_j>` readout. Photonic status: `PARTIAL_MERLIN_TRANSLATION`, because
the partial trace that discards the input register cannot be expressed through
MerLin's public API (see `lib/photonic.py`).


In [7]:
import math

from lib.photonic import PhotonicReservoir

photonic_rows = []
for divisor in (1, 8):
    reservoir = PhotonicReservoir(
        len(QR1_FEATURES), N_LAGS, seed=0, n_modes=10, n_photons=3,
        readout="mode_expectations", ensemble=False,
        encoding_scale=math.pi / divisor,
    )
    matrix = np.zeros((len(normalised), reservoir.n_readout))
    matrix[N_LAGS:] = reservoir.evaluate(windows[N_LAGS:])
    forecast = denormalise_log_rv(
        rolling_ridge_forecast(matrix, target, train_slices, origins, delta=1e-8)
    )
    photonic_rows.append({
        "encoding scale": f"pi/{divisor}", "readout width": reservoir.n_readout,
        "MSE": mse(forecast, actual), "QLIKE": qlike(forecast, actual),
    })

for key in ("computation_space", "detector_model", "n_modes", "n_photons",
            "input_state", "n_frozen_circuit_parameters", "n_trainable_parameters"):
    print(f"{key:32s} {reservoir.metadata[key]}")
pd.DataFrame(photonic_rows).round(4)


computation_space                UNBUNCHED
detector_model                   threshold (unbunched subspace)
n_modes                          10
n_photons                        3
input_state                      [1, 0, 0, 1, 0, 0, 1, 0, 0, 0]
n_frozen_circuit_parameters      360
n_trainable_parameters           10


,encoding scale,readout width,MSE,QLIKE
0,pi/1,10,0.1874,2.5436
1,pi/8,10,0.1582,4.8387


Accuracy improves as the encoding phase scale shrinks toward the linear limit — a
phase shifter is `2*pi`-periodic while `RY(pi x)` is `4*pi`-periodic, so copying
the gate-model scale folds half the feature range.

The two rows above are a *single* mesh seed, so they sit well above the sweep
means: photonic draws vary by an order of magnitude more than qubit draws
(SD 0.028-0.043 versus 0.003). Over the full sweep (25 mesh seeds x 4 scales x 2
variants, `configs/photonic.json`) the mean falls from 0.2070 at scale `pi` to
0.1482 at `pi/8` for PQR1, and the photonic ensemble PQR2 reaches **0.1004**
under the paper's best-of-N protocol — slightly better than the qubit QR2's best
of 100 (0.1018) and than its published value (0.1030).

## 6. Bottom line

Every quantum number in the paper reproduces to three or four decimals, and the
comparative claim still fails: with the HAR indexing corrected and the classical
controls given the same selection budget, six classical models beat both quantum
reservoirs. See `README.md` for the full tables and the verdict.
